In [7]:
# JSD (Jensen–Shannon divergence) utilities for pandas DataFrames
# - Works for (1) raw rows (count rows) and (2) pre-aggregated rows (sum a count column like "ID")
# - Computes, for each target (e.g., V_No), the JSD between:
#     target's distribution over contexts (e.g., genre/category)
#   vs global distribution over contexts (same filter)
#
# Typical use (your file):
#   df = pd.read_csv("...csv")
#   out = jsd_by_target(
#       df,
#       target_col="V_No",
#       context_col="category",
#       weight_col="ID",        # <- pre-aggregated counts
#       filters={"EP_T": True, "V_No": lambda s: s > 0},
#   )

from __future__ import annotations
import numpy as np
import pandas as pd
from typing import Dict, Any, Callable, Optional, Union

# ----------------------------
# Core math
# ----------------------------
# Helper: normalize counts to probabilities safely
# (nonnegative, sum=1, with optional smoothing)
def _safe_prob(x: np.ndarray, smooth: float = 0.0) -> np.ndarray:
    """Convert nonnegative counts to a probability vector (sum=1)."""
    x = np.asarray(x, dtype=float)
    if np.any(x < 0):
        raise ValueError("Counts must be nonnegative.")
    if smooth and smooth > 0:
        x = x + smooth
    s = x.sum()
    if s == 0:
        # all zeros -> return zeros (caller can decide how to handle)
        return np.zeros_like(x)
    return x / s

def jsd(p: np.ndarray, q: np.ndarray, base: float = 2.0, smooth: float = 0.0) -> float:
    """
    Jensen–Shannon divergence JSD(p||q).
    p, q can be counts or probabilities (nonnegative); internally normalized.
    smooth: add smoothing to counts BEFORE normalization (Laplace-like).
    """
    p = _safe_prob(p, smooth=smooth)
    q = _safe_prob(q, smooth=smooth)

    # if both are all-zero (shouldn't happen after normalization unless sums were 0)
    if p.sum() == 0 and q.sum() == 0:
        return 0.0

    m = 0.5 * (p + q)

    def _kl(a, b):
        # Kullback–Leibler divergence:  D_KL(a||b) = sum a_i * log(a_i / b_i)
        # KL(a||b) with 0*log(0/.) treated as 0
        mask = a > 0    # avoid log(0), ignore a_i = 0 terms
        return float(np.sum(a[mask] * np.log(a[mask] / b[mask])))   

    val = 0.5 * _kl(p, m) + 0.5 * _kl(q, m)
    if base is not None:
        val = val / np.log(base)
    return float(val)

# ----------------------------
# Counting / aggregation
# ----------------------------
FilterValue = Union[Any, Callable[[pd.Series], Union[pd.Series, np.ndarray]]]

def apply_filters(df: pd.DataFrame, filters: Optional[Dict[str, FilterValue]] = None) -> pd.DataFrame:
    """
    filters examples:
      {"EP_TT": True, "V_No": lambda s: s > 0}
      {"category": ["강의", "낭독"]}   # keep only these
    """
    if not filters:
        return df

    mask = pd.Series(True, index=df.index)
    for col, rule in filters.items():
        if col not in df.columns:
            raise KeyError(f"Missing filter column: {col}")

        s = df[col]
        if callable(rule):  # function taking Series, returning boolean Series/array, e.g., lambda s: s > 0
            m = rule(s)
            mask &= pd.Series(m, index=df.index)
        elif isinstance(rule, (list, tuple, set)):  # list of allowed values, e.g., ["강의", "낭독"]
            mask &= s.isin(rule)
        else:                                       # exact match value, e.g., True
            mask &= (s == rule)

    return df.loc[mask].copy()

# Helper: get counts per context, either by rows or summing a weight column
def context_counts(
    df: pd.DataFrame,
    context_col: str,                   # e.g., "category"
    weight_col: Optional[str] = None,   # e.g., "ID"
    count_mode: str = "auto",           # "auto" | "rows" | "sum"
) -> pd.Series:
    """
    Returns a Series indexed by context with counts.
    - rows mode: count rows per context
    - sum mode: sum weight_col per context
    - auto: if weight_col provided -> sum, else -> rows
    """
    if context_col not in df.columns:
        raise KeyError(f"Missing context_col: {context_col}")

    if count_mode not in {"auto", "rows", "sum"}:
        raise ValueError("count_mode must be one of: auto, rows, sum")

    if count_mode == "auto":
        count_mode = "sum" if weight_col else "rows"

    if count_mode == "rows":
        return df.groupby(context_col, dropna=False).size()
    else:
        if not weight_col:
            raise ValueError("weight_col is required for count_mode='sum'")
        if weight_col not in df.columns:
            raise KeyError(f"Missing weight_col: {weight_col}")
        return df.groupby(context_col, dropna=False)[weight_col].sum()

# ----------------------------
# Main: JSD by target
# ----------------------------
def jsd_by_target(
    df: pd.DataFrame,
    target_col: str,               # e.g., "V_No"
    context_col: str,            # e.g., "category"
    weight_col: Optional[str] = None,
    count_mode: str = "auto",
    filters: Optional[Dict[str, FilterValue]] = None,
    smooth: float = 0.0,
    base: float = 2.0,

    context_min_total: Optional[float] = None,  # drop low-frequency context types globally
    context_top_n: Optional[int] = None,        # or keep top-N context types globally

    target_min_total: float = 1.0,  # minimum total for each target after filtering
    target_min_context_types: Optional[int] = 0,    # drop targets with too few context types

    return_vectors: bool = False,
) -> pd.DataFrame:
    """
    For each target (e.g., V_No), compute JSD between:
      p = target's distribution over contexts
      q = global distribution over contexts (same filtered df)
    using either row counts or summed weights.
    context_min_total: drop low-frequency context types globally (before computing q).
    context_top_n: keep only top-N context types globally (before computing q).
    target_min_total: drop targets whose total count < target_min_total (after filtering).
    target_min_context_types: drop targets with fewer than this many nonzero context types.
    return_vectors: if True, includes p_vector/q_vector columns (lists) for debugging/export.
    """
    if target_col not in df.columns:
        raise KeyError(f"Missing target_col: {target_col}")
    if context_col not in df.columns:
        raise KeyError(f"Missing context_col: {context_col}")

    
    # -------------------------
    # 0) Apply filters
    d = apply_filters(df, filters=filters)
    if d.empty:
        return pd.DataFrame(columns=[target_col, "total", "jsd", "similarity_1_minus_jsd"]) # empty
    # -------------------------
    # 1) Build GLOBAL context counts (for vocabulary selection and Q)
    q_counts = context_counts(d, context_col=context_col, weight_col=weight_col, count_mode=count_mode)

    # -------------------------
    # 2) Apply context vocabulary constraint (GLOBAL)
    if context_min_total is not None:
        q_counts = q_counts[q_counts >= context_min_total] # drop low-frequency contexts
    if context_top_n is not None:
        q_counts = q_counts.sort_values(ascending=False).head(context_top_n) # keep top-N contexts
    
    # -------------------------
    # 3) Global Q vector restricted to kept contexts: define a fixed context order (so every target gets same-length vectors)
    contexts = q_counts.index.tolist() # kept contexts

    if len(contexts) == 0: # If everything was filtered out, fail loudly
        raise ValueError("No contexts left after applying context_min_total/context_top_n.")
    q_vec = q_counts.reindex(contexts, fill_value=0).to_numpy(dtype=float) # align to contexts,
    # q_vec is global context distribution

    # -------------------------
    # 4) Per-target JSD
    rows = []
    for tgt, g in d.groupby(target_col, dropna=False):
        p_counts = context_counts(g, context_col=context_col, weight_col=weight_col, count_mode=count_mode)
        p_vec = p_counts.reindex(contexts, fill_value=0).to_numpy(dtype=float)  # align to contexts 
        total = float(p_vec.sum())
        
        # ✅ target 총빈도 제약
        if total < target_min_total:   # drop targets whose total count < target_min_total
            continue
        # ✅ target의 최소 출현 context 유형 제약
        nonzero_contexts = np.sum(p_vec > 0)
        if nonzero_contexts < target_min_context_types:
            continue

        j = jsd(p_vec, q_vec, base=base, smooth=smooth)
        r = {
            target_col: tgt,
            "total": total,
            "jsd": j,
            "similarity_1_minus_jsd": 1.0 - j,
            "n_contexts_kept": len(contexts),
        }
        if return_vectors:
            r["p_vector"] = p_vec.tolist()
            r["q_vector"] = q_vec.tolist()
            r["contexts"] = contexts
        rows.append(r)

    out = pd.DataFrame(rows).sort_values(["jsd", "total"], ascending=[True, False]).reset_index(drop=True)
    return out


In [ ]:
# ----------------------------
# 실행
# ----------------------------
from pathlib import Path
from datetime import datetime

CSV_PATH = r"csv\통계용\V_VX_E_category-었-결합_2025-12-19_17-53.csv"
path = r"csv\통계용\V_VX_E_category-었-결합_2025-12-19_17-53.csv"
df = pd.read_csv(path)

UNIT_COL ="V_No"
df["EPxGenre"] = list(zip(df["category"], df["EP_T"])) #context combining category and EP_T
#CONTEXT_COL="category"
CONTEXT_COL="EPxGenre" 
WEIGHT_COL="ID"
"""
filters examples:
    {"EP_TT": True, "V_No": lambda s: s > 0}
    {"category": ["강의", "낭독"]}   # keep only these
"""
Filters = {
    #"EP_T": True,               # <- only rows where EP_T is True (e.g., -었- 결합)
    "EN_label": "EF",           # EF만 사용
    "EN_No": lambda s: (s > 1100) & (s < 1102), #다, 는다 한정
    "copus": 2,
}

if __name__ == "__main__":
    
    # (A) Pre-aggregated version: sum the "ID" column
    out_sumID = jsd_by_target(
        df,
        target_col=UNIT_COL,
        context_col=CONTEXT_COL,
        weight_col=WEIGHT_COL,
        count_mode="sum",   # "auto" | "rows" | "sum"
        filters=Filters,
        smooth=0.5,     # optional smoothing for sparse verbs
        base=2.0,       # optional log base
        context_min_total= 50,  # drop low-frequency context types globally
        # context_top_n = 2000,        # or keep top-N context types globally

        target_min_total = 100,  # minimum total for each target after filtering
        #target_min_context_types = 2,    # drop targets with too few context types
        return_vectors = True
        
    )
    print(out_sumID.head(20))

    #---- file name settings ----  
    SAVE_DIR = r"csv\결과값"   

    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
    out_file = f'JSD_{UNIT_COL}_by_{CONTEXT_COL}.csv'

    print(f"***저장합니다.:    {datetime.now()}***")
    print(f"file name: {timestamp}_{out_file}")
    out_sumID.to_csv(SAVE_DIR + f"\{timestamp}_{out_file}", index=False, encoding="utf-8-sig")

 



<string>:58: SyntaxWarning: invalid escape sequence '\{'
<>:58: SyntaxWarning: invalid escape sequence '\{'
<string>:58: SyntaxWarning: invalid escape sequence '\{'
<>:58: SyntaxWarning: invalid escape sequence '\{'
C:\Users\yu2hy\AppData\Local\Temp\ipykernel_26640\774016914.py:58: SyntaxWarning: invalid escape sequence '\{'
  out_sumID.to_csv(SAVE_DIR + f"\{timestamp}_{out_file}", index=False, encoding="utf-8-sig")


    V_No    total       jsd  similarity_1_minus_jsd  n_contexts_kept  \
0    105   5363.0  0.016253                0.983747               18   
1    103  17922.0  0.017397                0.982603               18   
2   3046    394.0  0.017943                0.982057               18   
3    102  29094.0  0.017967                0.982033               18   
4   2999   7638.0  0.018056                0.981944               18   
5   3094    165.0  0.018383                0.981617               18   
6   3026   1119.0  0.019354                0.980646               18   
7    106   6282.0  0.021247                0.978753               18   
8   3076    418.0  0.022418                0.977582               18   
9   2004  23890.0  0.025436                0.974564               18   
10  2071    183.0  0.026953                0.973047               18   
11  3022   6015.0  0.027094                0.972906               18   
12  3001  26169.0  0.027201                0.972799             

In [ ]:
if __name__ == "__main__":

    # (B) Raw-row version: count rows per category (ignore ID even if present)
    out_rows = jsd_by_target(
        df,
        target_col="V_No",
        context_col="category",
        weight_col=None,
        count_mode="rows",  # "auto" | "rows" | "sum"
        filters={"EP_TT": True, "V_No": lambda s: s > 0},
        smooth=0.5,
        min_total=10,
    )
    print(out_rows.head(20))
